# Autodata 3.45 — extracție în cloud (versiunea 2, fără Drive mount)

Această versiune rezolvă eroarea **OSError [Errno 107] Transport endpoint is not connected**: **nu folosește deloc `drive.mount()`**.

1. Apasă **Runtime → Run all** și autentifică-te în **același cont Google** în care este `Autodata__3.45.rar`.
2. Notebookul citește arhiva direct prin **Google Drive API**, în tranșe de 32 MiB, cu reluări la erorile temporare. Nu descarcă nimic pe telefon.
3. Verifică dimensiunea și MD5 (dacă este furnizat), testează arhiva, extrage fără să execute fișierele.
4. Încarcă fișierele extrase direct în **My Drive/Autodata_3_45_EXTRACTED**, folosind upload reluabil prin API, apoi creează `INVENTAR.json` și jurnale.

Arhiva originală rămâne neschimbată. Trebuie să existe spațiu suficient pe discul temporar Colab și în Drive. Dacă apare o solicitare pentru permisiuni Google Drive API, aprobă doar pentru acest notebook de încredere. **Nu publica parole, tokenuri sau arhiva.** Dacă runtime-ul Colab se resetează, discul temporar Colab este pierdut și descărcarea începe de la zero.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth, os, re, json, hashlib, shutil, subprocess, getpass, time
from pathlib import Path
from datetime import datetime, timezone
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError
from google.auth.transport.requests import AuthorizedSession

# Fișierul exact găsit în Google Drive, nu un link public.
SOURCE_ID = "1lC8pgAMU4RMkcatQClkzNnABJZpsKxZE"
FOLDER_NAME = "Autodata_3_45_EXTRACTED"
CACHE = Path("/content/autodata_work")
RAR = CACHE / "Autodata__3.45.rar"
PART = CACHE / "Autodata__3.45.rar.part"
EXTRACT = CACHE / "extracted"
LOGS = CACHE / "logs"
CHUNK = 32 * 1024 * 1024
MAX_UNPACKED_GIB = 20

credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
metadata = drive_api.files().get(
    fileId=SOURCE_ID, fields="id,name,size,mimeType,md5Checksum,capabilities/canDownload"
).execute()
if metadata.get("capabilities", {}).get("canDownload") is False:
    raise RuntimeError("Contul autentificat nu poate descărca arhiva.")
SOURCE = Path(metadata["name"])
size = int(metadata["size"])
expected_md5 = metadata.get("md5Checksum")
CACHE.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)
print("Fișier Google Drive:", SOURCE.name)
print("Dimensiune:", round(size / 2**30, 3), "GiB")
print("Spațiu temporar disponibil:", round(shutil.disk_usage("/content").free / 2**30, 2), "GiB")
if size < 2 * 10**9 or size > 3 * 10**9:
    raise RuntimeError("Dimensiunea arhivei nu corespunde fișierului identificat anterior.")

In [ ]:
# Descărcare exclusiv prin API — fără acces la /content/drive și fără FUSE.
session = AuthorizedSession(credentials)
url = "https://www.googleapis.com/drive/v3/files/" + SOURCE_ID + "?alt=media"
if not RAR.exists() or RAR.stat().st_size != size:
    start = PART.stat().st_size if PART.exists() else 0
    # Înlăturăm orice tranșă incompletă dintr-o încercare precedentă.
    start = (start // CHUNK) * CHUNK
    with PART.open("r+b" if PART.exists() else "wb") as f:
        f.truncate(start)
        while start < size:
            end = min(start + CHUNK, size) - 1
            expected = end - start + 1
            last_error = None
            for attempt in range(1, 9):
                response = None
                try:
                    response = session.get(
                        url, headers={"Range": f"bytes={start}-{end}"},
                        stream=True, timeout=(30, 180),
                    )
                    if response.status_code != 206:
                        if response.status_code in (429, 500, 502, 503, 504):
                            raise ConnectionError("Google API HTTP " + str(response.status_code))
                        raise RuntimeError(
                            f"Google API a returnat HTTP {response.status_code}, nu 206. "
                            "Verifică permisiunile Drive și contul Google."
                        )
                    cr = response.headers.get("Content-Range", "")
                    if not cr.startswith(f"bytes {start}-{end}/"):
                        raise RuntimeError("Răspuns Range neașteptat: " + cr)
                    f.seek(start)
                    f.truncate(start)
                    transferred = 0
                    for piece in response.iter_content(chunk_size=1024*1024):
                        if piece:
                            f.write(piece)
                            transferred += len(piece)
                    if transferred != expected:
                        raise ConnectionError(f"Tranșă incompletă: {transferred} din {expected}")
                    f.flush()
                    last_error = None
                    break
                except (OSError, ConnectionError, TimeoutError) as e:
                    last_error = e
                    f.seek(start)
                    f.truncate(start)
                    print(f"Conexiune întreruptă la {start/size:.0%}: încercarea {attempt}/8")
                    time.sleep(min(30, 2**attempt))
                finally:
                    if response is not None:
                        response.close()
            if last_error is not None:
                raise RuntimeError("Descărcarea a depășit numărul de reîncercări.") from last_error
            start = end + 1
            if start == size or start % (128*1024*1024) == 0:
                print(f"Descărcat din Google Drive API: {start/size:.0%}")
    PART.replace(RAR)
if RAR.stat().st_size != size:
    raise RuntimeError("Fișierul local are dimensiune incorectă.")

md5 = hashlib.md5()
sha = hashlib.sha256()
with RAR.open("rb") as f:
    for block in iter(lambda: f.read(8*1024*1024), b""):
        md5.update(block)
        sha.update(block)
if expected_md5 and md5.hexdigest().lower() != expected_md5.lower():
    raise RuntimeError("MD5 nu corespunde — arhiva locală nu va fi extrasă.")
SHA256 = sha.hexdigest()
print("Verificare copie: PASS; SHA256:", SHA256)

In [ ]:
# Test și extragere, fără executarea programelor din arhivă.
if not shutil.which("7z"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)

def sevenzip(action, password=None):
    args = ["7z", action, "-y"]
    if password:
        args.append("-p" + password)
    if action == "x":
        args.append("-o" + str(EXTRACT))
    args.append(str(RAR))
    with (LOGS / f"{action}.log").open("w", encoding="utf-8") as log:
        return subprocess.run(args, stdin=subprocess.DEVNULL, stdout=log,
                              stderr=subprocess.STDOUT, timeout=7200).returncode

password = None
if sevenzip("t") != 0:
    print((LOGS/"t.log").read_text(errors="replace")[-1500:])
    if input("Arhiva are parolă? (da/nu): ").strip().lower() in ("da", "d", "yes", "y"):
        password = getpass.getpass("Parola RAR: ")
        if sevenzip("t", password) != 0:
            raise RuntimeError("Arhiva nu a trecut testul, nici cu parola indicată.")
    else:
        raise RuntimeError("Arhiva nu a trecut testul de integritate.")

args = ["7z", "l", "-slt"]
if password:
    args.append("-p" + password)
args.append(str(RAR))
listing = subprocess.run(args, capture_output=True, text=True, stdin=subprocess.DEVNULL,
                         errors="replace", check=True, timeout=1800).stdout
(LOGS / "listing.txt").write_text(listing, encoding="utf-8")
detail = listing.split("----------", 1)[-1]
expanded = sum(int(n) for n in re.findall(r"(?m)^Size = (\d+)$", detail))
entries = re.findall(r"(?m)^Path = (.+)$", detail)
for name in entries:
    clean = name.replace("\\", "/")
    path = Path(clean)
    if path.is_absolute() or ".." in path.parts or "\x00" in name:
        raise RuntimeError("Arhiva include căi nesigure. Extragerea este blocată.")
print("Dimensiune estimată după extragere:", round(expanded/2**30, 2), "GiB")
if expanded > MAX_UNPACKED_GIB * 2**30:
    raise RuntimeError(f"Arhiva depășește plafonul {MAX_UNPACKED_GIB} GiB.")
if expanded and shutil.disk_usage("/content").free < expanded + 256*2**20:
    raise RuntimeError("Spațiu insuficient pe discul temporar Colab.")
EXTRACT.mkdir(exist_ok=True)
if sevenzip("x", password) != 0:
    print((LOGS/"x.log").read_text(errors="replace")[-2000:])
    raise RuntimeError("Extragerea a eșuat.")
print("Extragerea a reușit.")

In [ ]:
# Încărcarea fișierelor în Drive se face TOT prin API, nu prin drive.mount().
records = []
for root, dirs, names in os.walk(EXTRACT, followlinks=False):
    rootp = Path(root)
    dirs[:] = [d for d in dirs if not (rootp/d).is_symlink()]
    for name in names:
        item = rootp / name
        if item.is_symlink() or not item.is_file():
            continue
        rel = item.relative_to(EXTRACT)
        records.append({"path": rel.as_posix(), "bytes": item.stat().st_size})
if not records:
    raise RuntimeError("Nu s-au găsit fișiere extrase.")
if len(records) > 120000:
    raise RuntimeError("Prea multe fișiere pentru încărcare automată.")
print("Fișiere extrase:", len(records))

def query_escape(s):
    return s.replace("\\", "\\\\").replace("'", "\\'")

rootid = drive_api.files().get(fileId="root", fields="id").execute()["id"]
folder_cache = {}
def get_or_create_folder(parent_id, name):
    key = (parent_id, name)
    if key in folder_cache:
        return folder_cache[key]
    q = f"'{parent_id}' in parents and name = '{query_escape(name)}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    existing = drive_api.files().list(q=q, fields="files(id,name),nextPageToken",pageSize=10).execute().get("files", [])
    if existing:
        folder_id = existing[0]["id"]
    else:
        folder_id = drive_api.files().create(
            body={"name":name,"mimeType":"application/vnd.google-apps.folder","parents":[parent_id]},
            fields="id").execute()["id"]
    folder_cache[key] = folder_id
    return folder_id

destid = get_or_create_folder(rootid, FOLDER_NAME)
def upload_one(local, parent_id):
    name = local.name
    q = f"'{parent_id}' in parents and name = '{query_escape(name)}' and trashed = false"
    matches = drive_api.files().list(q=q, fields="files(id,name,size,md5Checksum)",pageSize=10).execute().get("files",[])
    media = MediaFileUpload(str(local), mimetype="application/octet-stream",
                            resumable=True, chunksize=8*1024*1024)
    if matches:
        req = drive_api.files().update(fileId=matches[0]["id"], media_body=media, fields="id,size")
    else:
        req = drive_api.files().create(body={"name":name,"parents":[parent_id]},
                                       media_body=media, fields="id,size")
    result = None
    while result is None:
        _, result = req.next_chunk(num_retries=7)
    return result["id"]

for i, record in enumerate(records, 1):
    rel = Path(record["path"])
    folder = destid
    for part in rel.parts[:-1]:
        folder = get_or_create_folder(folder, part)
    upload_one(EXTRACT / rel, folder)
    if i % 100 == 0 or i == len(records):
        print(f"Încărcat în Drive prin API: {i}/{len(records)}")

manifest = {"archive":SOURCE.name,"archive_size":size, "sha256":SHA256,
            "created_utc":datetime.now(timezone.utc).isoformat(),
            "file_count":len(records),"files":records}
inventory = CACHE / "INVENTAR.json"
inventory.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
upload_one(inventory,destid)
invtext = CACHE / "INVENTAR.txt"
invtext.write_text("\n".join(f"{v['bytes']:>12} {v['path']}" for v in records),encoding="utf-8")
upload_one(invtext,destid)
for fname, output in [("t.log","verificare_7z.txt"),("x.log","extragere_7z.txt"),
                      ("listing.txt","listare_7z.txt")]:
    source = LOGS / fname
    target = CACHE / output
    shutil.copyfile(source, target)
    upload_one(target,destid)
print("TERMINAT: arhiva extrasă și salvată în Google Drive.")
print("Folder:", "https://drive.google.com/drive/folders/" + destid)
print("INVENTAR.json a fost încărcat cu succes.")